# Why capitals? -- two CPU-only tests (local)

The drop in context reliance is concentrated in country capitals. Answer length and parametric strength
are already ruled out. Two cheap hypotheses, tested on data we already have or can download:

1. **The trough model judges the passage as not answering the question.** At tulu's trough the model often
   continues its answer with "Is the answer in context? no" (a format from the Tulu training data).
   *Prediction if this explains the flips:* the pattern is specific to capitals, and much more frequent
   in flipping capitals than in capitals that stay context-following.
2. **The SFT data teaches capital facts.** "What is the capital of X?" is a very common template.
   *Prediction if this explains which capitals flip:* flipping capitals (their country and real capital)
   appear more often in that run's training data than stable capitals do.

Flip / stable groups are the eval-based ones from `flip_set.json` (capitals only).

In [ ]:
import json, re
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact, mannwhitneyu

PHASES = {"tulu": {"peak": 500, "trough": 3150, "recovery": 5000}, "alpaca": {"peak": 100, "trough": 700}}
STRATA = {"cap": "capitals", "bio": "biographies", "wf": "world facts"}

def load_eval(run, step):
    f = f"../../results/cpi-results/{run}-results/checkpoint_step{step}_metrics.json"
    return {r["item_id"]: r for r in json.load(open(f, encoding="utf-8"))["per_item"]}

flip_set = json.load(open("flip_set.json"))
GROUPS = {}
for run, fs in flip_set.items():
    flips = [r["item_id"] for g, rows in fs["groups"].items() if g.startswith("flip") for r in rows if r["stratum"] == "cap"]
    stable = [r["item_id"] for r in fs["groups"]["stable_ctx"] if r["stratum"] == "cap"]
    GROUPS[run] = {"flip": flips, "stable": stable}
    print(f"{run}: {len(flips)} flipping capitals, {len(stable)} stable capitals")

## Test 1 -- "Is the answer in context?" in the generated text

In [ ]:
IN_CONTEXT = re.compile(r"in context\?", re.I)
SAYS_NO = re.compile(r"in context\?\s*no", re.I)

rows = []
for run, phases in PHASES.items():
    for phase, step in phases.items():
        ev = load_eval(run, step)
        for s, name in STRATA.items():
            resp = [r["response"] for i, r in ev.items() if i.startswith(s + "_")]
            rows.append({"run": run, "phase": phase, "stratum": name,
                         "has 'in context?'": np.mean([bool(IN_CONTEXT.search(t)) for t in resp]),
                         "... followed by 'no'": np.mean([bool(SAYS_NO.search(t)) for t in resp])})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.0%}"))

print("\nAt the trough, capitals only: flipping vs stable")
for run in PHASES:
    ev = load_eval(run, PHASES[run]["trough"])
    a = sum(bool(SAYS_NO.search(ev[i]["response"])) for i in GROUPS[run]["flip"])
    b = sum(bool(SAYS_NO.search(ev[i]["response"])) for i in GROUPS[run]["stable"])
    nf, ns = len(GROUPS[run]["flip"]), len(GROUPS[run]["stable"])
    odds, p = fisher_exact([[a, nf - a], [b, ns - b]])
    print(f"  {run:6s}: says 'in context? no' -- flips {a}/{nf} ({a/nf:.0%}), stable {b}/{ns} ({b/ns:.0%})"
          f"   Fisher odds ratio {odds:.1f}, p = {p:.2g}")

## Test 2 -- how often each capital fact appears in the training data
For every capitals item (country X, real capital P), count the SFT training examples that mention
- X at all,
- X together with the word "capital",
- X together with P.

Word-boundary matches, case-insensitive. By the trough both runs had seen more than one full pass of their
data (tulu 3150 x 128 = 403k examples of 326k; alpaca 700 x 128 = 90k of 52k), so the whole dataset counts.
Countries with ambiguous names (Georgia, Chad, Jordan...) add noise, equally to flips and stable items.

In [ ]:
from huggingface_hub import hf_hub_download

SFT = {"alpaca": ("tatsu-lab/alpaca", ["data/train-00000-of-00001-a09b74b3ef9c3b56.parquet"]),
       "tulu":   ("allenai/tulu-v2-sft-mixture", ["data/train-00000-of-00003-99ee8754042a69f6.parquet",
                                                  "data/train-00001-of-00003-278198836de5994c.parquet",
                                                  "data/train-00002-of-00003-a52de323599d586a.parquet"])}

def example_texts(run):
    # yields one lower-cased string per training example
    repo, files = SFT[run]
    for f in files:
        df = pd.read_parquet(hf_hub_download(repo, f, repo_type="dataset"))
        if run == "alpaca":
            texts = (df["instruction"] + " " + df["input"] + " " + df["output"]).str.lower()
        else:
            texts = df["messages"].map(lambda msgs: " ".join(m["content"] for m in msgs).lower())
        yield from texts

# capitals items: country from the question, real capital = parametric answer
items = json.load(open("../../dataset/conflict_eval_unified.json", encoding="utf-8"))
items = items["items"] if isinstance(items, dict) else items
CAPS = {}
for it in items:
    if it["item_id"].startswith("cap_"):
        m = re.search(r"capital (?:city )?of (.+?)\?", it["question"])
        if m:
            CAPS[it["item_id"]] = {"country": m.group(1).strip().lower(), "capital": it["parametric_answer"].strip().lower()}
print(len(CAPS), "capitals items; e.g.", list(CAPS.values())[:3])

word = lambda s: re.compile(r"\b" + re.escape(s) + r"\b")
COUNTRY_RE = re.compile(r"\b(" + "|".join(re.escape(c) for c in sorted({v["country"] for v in CAPS.values()}, key=len, reverse=True)) + r")\b")
CAPITAL_RE = {v["country"]: word(v["capital"]) for v in CAPS.values()}

In [ ]:
# Count (a few minutes for tulu). Saved to CSV so this only runs once.
import os, time
COUNTS_FILE = "why_capitals_sft_counts.csv"

if os.path.exists(COUNTS_FILE):
    counts = pd.read_csv(COUNTS_FILE)
else:
    out = []
    for run in ["alpaca", "tulu"]:
        t0 = time.time()
        n_country, n_capital_word, n_pair, n_examples = {}, {}, {}, 0
        for text in example_texts(run):
            n_examples += 1
            for c in set(COUNTRY_RE.findall(text)):
                n_country[c] = n_country.get(c, 0) + 1
                if "capital" in text:
                    n_capital_word[c] = n_capital_word.get(c, 0) + 1
                if CAPITAL_RE[c].search(text):
                    n_pair[c] = n_pair.get(c, 0) + 1
        for item_id, v in CAPS.items():
            c = v["country"]
            out.append({"run": run, "item_id": item_id, "country": c, "capital": v["capital"],
                        "with_country": n_country.get(c, 0), "with_country_and_word_capital": n_capital_word.get(c, 0),
                        "with_country_and_its_capital": n_pair.get(c, 0)})
        print(f"{run}: {n_examples} examples counted in {time.time() - t0:.0f}s")
    counts = pd.DataFrame(out)
    counts.to_csv(COUNTS_FILE, index=False)
print(counts.head())

In [ ]:
# Flipping vs stable capitals: do the flipping ones appear more in that run's training data?
MEASURES = ["with_country", "with_country_and_word_capital", "with_country_and_its_capital"]
for run in ["tulu", "alpaca"]:
    c = counts[counts.run == run].set_index("item_id")
    print(f"\n{run} SFT data")
    for m in MEASURES:
        f, s = c.loc[[i for i in GROUPS[run]["flip"] if i in c.index], m], c.loc[[i for i in GROUPS[run]["stable"] if i in c.index], m]
        p = mannwhitneyu(f, s).pvalue
        print(f"  {m:32s} median flips {f.median():6.0f}  vs stable {s.median():6.0f}   "
              f"(means {f.mean():7.1f} vs {s.mean():7.1f})   Mann-Whitney p = {p:.2g}")

## Reading the results
- Test 1 supports hypothesis 1 only if the pattern is capitals-specific **and** clearly more frequent in
  flipping than stable capitals, in **both** runs. Alpaca never produces this format, so at most it can
  explain tulu.
- Test 2 supports hypothesis 2 if flipping capitals appear clearly more often (small p, a sizeable gap in
  medians) in **both** runs' data. No difference means the SFT frequency of a fact does not decide which
  capitals flip.